In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D9 — Movimento Fisiológico da População de Portugal, 1925
# ============================================================

!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-por tesseract-ocr-fra -qq

!pip install -q pymupdf pdf2image pytesseract pillow

import hashlib
import json
import math
import platform
import re
import sys
from collections import Counter
from pathlib import Path

import fitz
import pandas as pd
import pytesseract
from google.colab import files
from pdf2image import convert_from_path
from PIL import Image, ImageFilter, ImageOps

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Selecting previously unselected package tesseract-ocr-fra.
Preparing to unpack .../tesseract-ocr-fra_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-fra (1:4.1.0-2) ...
Selecting previously unselected package tesseract-ocr-por.
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-fra (1:4.1.0-2) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

SOURCE_FORMAT = "PDF"
EXPECTED_PAGE_COUNT = 8
EXPECTED_REFERENCE_RECORD_COUNT = 19

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
}

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Reporting Period"
]

OUTPUT_DIR = Path("outputs_D9_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_VALUES_PATH = OUTPUT_DIR / "D9_reference_values.csv"
REFERENCE_VALUES_JSON_PATH = OUTPUT_DIR / "D9_reference_values.json"
REFERENCE_SCHEMA_PATH = OUTPUT_DIR / "D9_reference_schema.json"
REFERENCE_SUMMARY_PATH = OUTPUT_DIR / "D9_reference_summary.json"
REFERENCE_INTEGRITY_PATH = OUTPUT_DIR / "D9_reference_integrity.json"
DOCUMENT_CHARACTERISATION_PATH = OUTPUT_DIR / "D9_document_characterisation.json"
QUALITY_EVIDENCE_PATH = OUTPUT_DIR / "D9_quality_evidence.json"
OCR_RESULTS_PATH = OUTPUT_DIR / "D9_ocr_results.csv"
EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D9_extraction_schema.json"
)
EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D9_extraction_task.txt"
)
INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D9_indicator_assessment.csv"
)
DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D9_dimension_assessment.csv"
)
REFERENCE_METADATA_PATH = (
    OUTPUT_DIR / "D9_reference_metadata.json"
)
IMAGE_METADATA_PATH = OUTPUT_DIR / "D9_image_metadata.csv"
OCR_DIAGNOSTICS_PATH = OUTPUT_DIR / "D9_ocr_diagnostics.json"
NATIVE_PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D9_native_page_characterisation.csv"
)

print("Document:", DOCUMENT_ID)
print("Expected PDF pages:", EXPECTED_PAGE_COUNT)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)


Document: D9
Expected PDF pages: 8
Expected reference records: 19
Output directory: outputs_D9_stage1


In [3]:
# ============================================================
# 2. Upload the original D9 PDF
# ============================================================

print("Upload the original D9 PDF.")

uploaded = files.upload()

pdf_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError(
        "Upload exactly one PDF source document."
    )

SOURCE_PATH = pdf_paths[0]

print("Source file:", SOURCE_PATH.name)
print("Source size:", f"{SOURCE_PATH.stat().st_size:,} bytes")


Upload the original D9 PDF.


Saving D9 - EMovimentoFisiológico1925.pdf to D9 - EMovimentoFisiológico1925.pdf
Source file: D9 - EMovimentoFisiológico1925.pdf
Source size: 296,008 bytes


In [4]:
# ============================================================
# 3. Hashing utility and PDF inspection
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

native_page_rows = []

for page_number, page in enumerate(pdf_document, start=1):
    native_text = page.get_text("text") or ""

    native_page_rows.append({
        "Page Number": page_number,
        "Native Character Count": len(native_text),
        "Native Word Count": len(native_text.split()),
        "Native Text Extractable": bool(native_text.strip()),
        "Width": float(page.rect.width),
        "Height": float(page.rect.height),
        "PDF Rotation": int(page.rotation)
    })


native_page_df = pd.DataFrame(native_page_rows)

TOTAL_NATIVE_CHARACTERS = int(
    native_page_df["Native Character Count"].sum()
)

TEXT_EXTRACTABLE = bool(
    TOTAL_NATIVE_CHARACTERS > 100
)

OCR_REQUIRED = not TEXT_EXTRACTABLE

print("Source SHA-256:", SOURCE_SHA256)
print("Page count:", PAGE_COUNT)
print("Page count valid:", PAGE_COUNT_VALID)
print("Native characters:", TOTAL_NATIVE_CHARACTERS)
print("Text extractable:", TEXT_EXTRACTABLE)
print("OCR required:", OCR_REQUIRED)

display(native_page_df)

if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, found {PAGE_COUNT}."
    )


Source SHA-256: a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1
Page count: 8
Page count valid: True
Native characters: 0
Text extractable: False
OCR required: True


,Page Number,Native Character Count,Native Word Count,Native Text Extractable,Width,Height,PDF Rotation
0,1,0,0,False,576.00000,840.599976,0
1,2,0,0,False,576.00000,840.599976,0
2,3,0,0,False,576.00000,840.599976,0
3,4,0,0,False,581.76001,841.320007,0
4,5,0,0,False,576.00000,840.599976,0
5,6,0,0,False,576.00000,840.599976,0
6,7,0,0,False,581.76001,841.320007,0
7,8,0,0,False,576.00000,840.599976,0


In [5]:
# ============================================================
# 4. Render PDF pages to images
# ============================================================

RENDER_DPI = 300

rendered_pages = convert_from_path(
    str(SOURCE_PATH),
    dpi=RENDER_DPI,
    fmt="png"
)

if len(rendered_pages) != EXPECTED_PAGE_COUNT:
    raise AssertionError(
        "Rendered page count does not match the PDF page count."
    )


image_metadata_rows = []

for page_number, image in enumerate(rendered_pages, start=1):
    image_path = OUTPUT_DIR / f"D9_page_{page_number}.png"
    image.save(image_path)

    orientation = (
        "landscape"
        if image.width > image.height
        else "portrait"
    )

    image_metadata_rows.append({
        "Page Number": page_number,
        "Image Width": image.width,
        "Image Height": image.height,
        "Orientation": orientation,
        "Rendered DPI": RENDER_DPI,
        "Image Path": str(image_path)
    })


image_metadata_df = pd.DataFrame(
    image_metadata_rows
)

display(image_metadata_df)


,Page Number,Image Width,Image Height,Orientation,Rendered DPI,Image Path
0,1,2400,3503,portrait,300,outputs_D9_stage1/D9_page_1.png
1,2,2400,3503,portrait,300,outputs_D9_stage1/D9_page_2.png
2,3,2400,3503,portrait,300,outputs_D9_stage1/D9_page_3.png
3,4,2424,3506,portrait,300,outputs_D9_stage1/D9_page_4.png
4,5,2400,3503,portrait,300,outputs_D9_stage1/D9_page_5.png
5,6,2400,3503,portrait,300,outputs_D9_stage1/D9_page_6.png
6,7,2424,3506,portrait,300,outputs_D9_stage1/D9_page_7.png
7,8,2400,3503,portrait,300,outputs_D9_stage1/D9_page_8.png


In [6]:
# ============================================================
# 5. OCR preprocessing and diagnostic extraction
# ============================================================

def prepare_for_ocr(image, rotate_degrees=0):
    working = image.copy()

    if rotate_degrees:
        working = working.rotate(
            rotate_degrees,
            expand=True,
            fillcolor="white"
        )

    working = ImageOps.grayscale(working)
    working = ImageOps.autocontrast(working)
    working = working.filter(
        ImageFilter.SHARPEN
    )

    return working


# Pages 7 and 8 are visually rotated in the supplied PDF.
# Both clockwise and counter-clockwise OCR alternatives are tested,
# and the result with the largest useful token count is retained.
ROTATION_CANDIDATES = {
    7: [90, -90],
    8: [90, -90]
}


def useful_token_count(text):
    return len(
        re.findall(
            r"[A-Za-zÀ-ÿ]{2,}|\d+(?:[.,]\d+)?",
            text
        )
    )


ocr_rows = []
ocr_page_texts = {}

for page_number, image in enumerate(rendered_pages, start=1):

    candidate_rotations = ROTATION_CANDIDATES.get(
        page_number,
        [0]
    )

    candidates = []

    for rotation in candidate_rotations:
        prepared = prepare_for_ocr(
            image,
            rotate_degrees=rotation
        )

        text = pytesseract.image_to_string(
            prepared,
            lang="por+fra",
            config="--oem 1 --psm 6"
        )

        candidates.append({
            "rotation": rotation,
            "text": text,
            "useful_tokens": useful_token_count(text)
        })

    best_candidate = max(
        candidates,
        key=lambda item: item["useful_tokens"]
    )

    ocr_text = best_candidate["text"]
    ocr_page_texts[page_number] = ocr_text

    ocr_rows.append({
        "Page Number": page_number,
        "Selected Rotation": best_candidate["rotation"],
        "OCR Character Count": len(ocr_text),
        "OCR Word Count": len(ocr_text.split()),
        "Useful Token Count": best_candidate["useful_tokens"],
        "Numeric Token Count": len(
            re.findall(
                r"(?<!\w)[+-]?\d[\d. ]*(?:,\d+)?",
                ocr_text
            )
        ),
        "OCR Text": ocr_text
    })


ocr_results_df = pd.DataFrame(ocr_rows)
FULL_OCR_TEXT = "\n".join(
    ocr_page_texts[page_number]
    for page_number in range(1, EXPECTED_PAGE_COUNT + 1)
)

ocr_numeric_tokens = re.findall(
    r"(?<!\w)[+-]?\d[\d. ]*(?:,\d+)?",
    FULL_OCR_TEXT
)

ocr_word_count = len(
    FULL_OCR_TEXT.split()
)

ocr_numeric_token_to_word_ratio = (
    len(ocr_numeric_tokens)
    / ocr_word_count
    if ocr_word_count
    else 0
)


display(
    ocr_results_df[
        [
            "Page Number",
            "Selected Rotation",
            "OCR Character Count",
            "OCR Word Count",
            "Useful Token Count",
            "Numeric Token Count"
        ]
    ]
)


,Page Number,Selected Rotation,OCR Character Count,OCR Word Count,Useful Token Count,Numeric Token Count
0,1,0,240,45,29,2
1,2,0,57,9,9,0
2,3,0,3443,760,443,40
3,4,0,3882,864,479,30
4,5,0,2802,493,382,25
5,6,0,4690,835,629,295
6,7,90,5537,1147,1023,135
7,8,90,5044,1032,990,119


In [7]:
# ============================================================
# 6. OCR diagnostics and source-marker checks
# ============================================================

OCR_MARKER_PATTERNS = {
    "publication_title":
        r"Movimento\s+Fisiol[oó]gico",
    "reference_year":
        r"1925",
    "publication_year":
        r"1929",
    "index_heading":
        r"[ÍI]NDICE|TABLE\s+DES\s+MATI",
    "tabela_i":
        r"TABELA\s+I\b",
    "tabela_ii":
        r"TABELA\s+II\b",
    "taxas_demograficas":
        r"TAXAS\s+DEMOGR",
    "portugal":
        r"PORTUGAL",
    "1911":
        r"1911",
    "1920":
        r"1920"
}

ocr_marker_status = {
    marker: bool(
        re.search(
            pattern,
            FULL_OCR_TEXT,
            flags=re.IGNORECASE
        )
    )
    for marker, pattern
    in OCR_MARKER_PATTERNS.items()
}

ocr_marker_count = sum(
    ocr_marker_status.values()
)

ocr_dependency_confirmed = (
    OCR_REQUIRED
    and ocr_marker_count >= 6
)


OCR_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "native_text_characters": TOTAL_NATIVE_CHARACTERS,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "render_dpi": RENDER_DPI,
    "ocr_languages": ["por", "fra"],
    "rotated_pages": [7, 8],
    "ocr_marker_status": ocr_marker_status,
    "ocr_marker_count": ocr_marker_count,
    "ocr_dependency_confirmed": ocr_dependency_confirmed,
    "ocr_numeric_token_count":
        len(ocr_numeric_tokens),
    "ocr_word_count":
        ocr_word_count,
    "ocr_numeric_token_to_word_ratio":
        round(
            ocr_numeric_token_to_word_ratio,
            3
        ),
    "ocr_is_reference_source": False,
    "notes": (
        "OCR is used for document diagnostics only. Reference values "
        "are manually grounded in the rendered PDF pages."
    )
}

print(
    json.dumps(
        OCR_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

if not OCR_REQUIRED:
    print(
        "Warning: native text was detected. Review whether the supplied "
        "file differs from the expected image-based D9 PDF."
    )

if ocr_marker_count < 6:
    print(
        "Warning: OCR marker recall is low. This does not invalidate "
        "the manually constructed reference values."
    )


{
  "document_id": "D9",
  "source_file": "D9 - EMovimentoFisiológico1925.pdf",
  "source_file_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "page_count": 8,
  "page_count_valid": true,
  "native_text_characters": 0,
  "text_extractable": false,
  "ocr_required": true,
  "render_dpi": 300,
  "ocr_languages": [
    "por",
    "fra"
  ],
  "rotated_pages": [
    7,
    8
  ],
  "ocr_marker_status": {
    "publication_title": true,
    "reference_year": true,
    "publication_year": false,
    "index_heading": true,
    "tabela_i": true,
    "tabela_ii": false,
    "taxas_demograficas": true,
    "portugal": true,
    "1911": true,
    "1920": true
  },
  "ocr_marker_count": 8,
  "ocr_dependency_confirmed": true,
  "ocr_numeric_token_count": 646,
  "ocr_word_count": 5185,
  "ocr_numeric_token_to_word_ratio": 0.125,
  "ocr_is_reference_source": false,
  "notes": "OCR is used for document diagnostics only. Reference values are manually grounded in the render

In [8]:
# ============================================================
# 7. Define the fixed D9 extraction task and schema
# ============================================================

EXTRACTION_TASK = """You are an information extraction assistant.

Extract the fixed set of bibliographic, index, statistical and structural
records represented in the supplied historical scanned publication
"Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925".

Treat the supplied document as the only source.

Return exactly 19 records:

- 5 Publication metadata
- 6 Index entry
- 5 Statistical value
- 3 Document structure

For each record return exactly:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Preserve Portuguese source wording, historical spelling, punctuation,
measurement scale, printed table identifiers and physical PDF page references.
Do not calculate, infer, derive, convert, repair or modernise source values.
Populate Value only when the source represents a distinct value for the
record. Do not duplicate wording from Topic or Description into Value.

Populate Reporting Period only when a year or period is explicitly
associated with the individual record. Do not assign the publication's
general reference year to a record solely because it occurs in the
1925 publication.
Return valid JSON using the exact field names defined in the
  extraction schema.
Do not include explanations before or after the JSON.
"""


REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": (
        "One fixed publication metadata item, selected index entry, "
        "selected Tabela I value or source-grounded structural feature"
    ),
    "expected_fields": EXPECTED_FIELDS,
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "fields": {
        "Category": "One of the four fixed D9 category labels",
        "Topic": "The metadata field, table identifier, indicator or structural feature",
        "Description": "Source-grounded description of the represented record",
        "Value": (
            "Explicit source value represented separately from the record "
            "description; use null when no distinct value is represented"
        ),
        "Reporting Period": (
            "Year or period explicitly associated with the individual record; "
            "use null when the period is only general document context"
        ),
        "Unit": "Represented unit or text, or null",
        "Source Location": "Physical PDF page and source element"
    },
    "value_types": [
        "number",
        "string",
        "null"
    ],
    "preservation_rules": [
        "Preserve Portuguese titles and table identifiers",
        "Preserve the reference year 1925 separately from publication year 1929",
        "Preserve the represented 1911 and 1920 periods",
        "Preserve printed numeric scale without deriving values",
        "Do not infer values from Tabela II",
        "Do not treat OCR text as authoritative where visual evidence differs",
        "Use physical PDF page references"
    ]
}

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "historical_publication_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": ["string", "null"]
        },

        "Topic": {
            "type": ["string", "null"]
        },

        "Description": {
            "type": ["string", "null"]
        },

        "Value": {
            "type": [
                "string",
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": ["string", "null"]
        },

        "Reporting Period": {
            "type": ["string", "null"]
        },

        "Source Location": {
            "type": ["string", "null"]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Topic":
                    "string or null",

                "Description":
                    "string or null",

                "Value":
                    "string, number or null",

                "Unit":
                    "string or null",

                "Reporting Period":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}

print(EXTRACTION_TASK)

print(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)


You are an information extraction assistant.

Extract the fixed set of bibliographic, index, statistical and structural
records represented in the supplied historical scanned publication
"Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925".

Treat the supplied document as the only source.

Return exactly 19 records:

- 5 Publication metadata
- 6 Index entry
- 5 Statistical value
- 3 Document structure

For each record return exactly:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Preserve Portuguese source wording, historical spelling, punctuation,
measurement scale, printed table identifiers and physical PDF page references.
Do not calculate, infer, derive, convert, repair or modernise source values.
Populate Value only when the source represents a distinct value for the
record. Do not duplicate wording from Topic or Description into Value.

Populate Reporting Period only when a year or period is explicitly
associated wi

In [9]:
# ============================================================
# 8. Construct the fixed 19-record reference dataset
# ============================================================

reference_records = [
    # --------------------------------------------------------
    # Publication metadata — physical PDF page 1
    # --------------------------------------------------------
    {
        "Category": "Publication metadata",
        "Topic": "Title",
        "Description": "Publication title",
        "Value": (
            "Estatística do Movimento Fisiológico "
            "da População de Portugal"
        ),
        "Unit": None,
        "Reporting Period": "1925",
        "Source Location": "PDF page 1 — Cover"
    },
    {
        "Category": "Publication metadata",
        "Topic": "Reference year",
        "Description": "Statistical reference year",
        "Value": 1925,
        "Unit": "year",
        "Reporting Period": "1925",
        "Source Location": "PDF page 1 — Cover"
    },
    {
        "Category": "Publication metadata",
        "Topic": "Publication year",
        "Description": "Printed publication year",
        "Value": 1929,
        "Unit": "year",
        "Reporting Period": "1929",
        "Source Location": "PDF page 1 — Imprint"
    },
    {
        "Category": "Publication metadata",
        "Topic": "Publisher",
        "Description": "Publication printer",
        "Value": "Imprensa Nacional",
        "Unit": None,
        "Reporting Period": "1929",
        "Source Location": "PDF page 1 — Imprint"
    },
    {
        "Category": "Publication metadata",
        "Topic": "Institution",
        "Description": "Responsible institution",
        "Value": (
            "Direcção Geral de Saúde — Portugal; "
            "Inspecção de Demografia e Estatística"
        ),
        "Unit": None,
        "Reporting Period": "1925",
        "Source Location": "PDF page 1 — Cover"
    },

    # --------------------------------------------------------
    # Selected index entries — physical PDF pages 3–5
    # --------------------------------------------------------
    {
        "Category": "Index entry",
        "Topic": "Tabela I",
        "Description": (
            "Área e População recenseada "
            "(I-XII-1911 e I-XII-1920), por distritos e sexos"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": "1911 and 1920",
        "Source Location": "PDF page 3 — Índice; printed page 1"
    },
    {
        "Category": "Index entry",
        "Topic": "Tabela II",
        "Description": (
            "População recenseada (I-XII-1911 e I-XII-1920), "
            "por distritos, idades e sexos"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": "1911 and 1920",
        "Source Location": "PDF page 3 — Índice; printed pages 2 a 5"
    },
    {
        "Category": "Index entry",
        "Topic": "Tabela III",
        "Description": (
            "Casamentos, divórcios, nascimentos e óbitos, por distritos "
            "e concelhos, com nascimentos por legitimidade e sexos "
            "e óbitos por sexos"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 3 — Índice; printed pages 6 a 12"
    },
    {
        "Category": "Index entry",
        "Topic": "Tabela XIV",
        "Description": "Óbitos, por idades e sexos, nos distritos",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 3 — Índice; printed pages 70 e 71"
    },
    {
        "Category": "Index entry",
        "Topic": "Tabela LVIII",
        "Description": (
            "Taxas do movimento fisiológico, referidas "
            "à população calculada"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 5 — Índice; printed pages 240 e 241"
    },
    {
        "Category": "Index entry",
        "Topic": "Tabela LIX",
        "Description": (
            "Excesso dos nascimentos sobre os óbitos e crescimento "
            "fisiológico da população, por distritos e por meses"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 5 — Índice; printed page 242"
    },

    # --------------------------------------------------------
    # Selected Tabela I values — physical PDF page 6
    # --------------------------------------------------------
    {
        "Category": "Statistical value",
        "Topic": "Portugal area",
        "Description": "Total area of Portugal",
        "Value": 91948.07,
        "Unit": "square kilometres",
        "Reporting Period": None,
        "Source Location": "PDF page 6 — Tabela I, Portugal row"
    },
    {
        "Category": "Statistical value",
        "Topic": "Portugal population 1911",
        "Description": "Total population of Portugal in 1911",
        "Value": 5960056,
        "Unit": "people",
        "Reporting Period": "1911",
        "Source Location": "PDF page 6 — Tabela I, Portugal row"
    },
    {
        "Category": "Statistical value",
        "Topic": "Portugal population 1920",
        "Description": "Total population of Portugal in 1920",
        "Value": 6032991,
        "Unit": "people",
        "Reporting Period": "1920",
        "Source Location": "PDF page 6 — Tabela I, Portugal row"
    },
    {
        "Category": "Statistical value",
        "Topic": "Portugal density 1920",
        "Description": "Population density of Portugal",
        "Value": 65.6,
        "Unit": "inhabitants per square kilometre",
        "Reporting Period": "1920",
        "Source Location": "PDF page 6 — Tabela I, Portugal row"
    },
    {
        "Category": "Statistical value",
        "Topic": "Portugal average annual population growth",
        "Description": (
            "Average annual population increase "
            "between 1911 and 1920"
        ),
        "Value": 1.36,
        "Unit": "per thousand inhabitants",
        "Reporting Period": "1911-1920",
        "Source Location": "PDF page 6 — Tabela I, Portugal row"
    },

    # --------------------------------------------------------
    # Source-grounded structural records
    # --------------------------------------------------------
    {
        "Category": "Document structure",
        "Topic": "Rotated table",
        "Description": (
            "Tabela II is presented in a rotated layout "
            "across two supplied PDF pages"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF pages 7–8 — Tabela II"
    },
    {
        "Category": "Document structure",
        "Topic": "Bilingual headings",
        "Description": (
            "Portuguese headings are accompanied "
            "by French translations"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF pages 3–8 — Index and tables"
    },
    {
        "Category": "Document structure",
        "Topic": "Historical typography",
        "Description": (
            "The publication uses historical typography "
            "and scanned tabular layouts"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF pages 1–8"
    },
]


reference_values_df = pd.DataFrame(
    reference_records,
    columns=EXPECTED_FIELDS
)

print("Reference records:", len(reference_values_df))

display(reference_values_df)


Reference records: 19


,Category,Topic,Description,Value,Unit,Reporting Period,Source Location
0,Publication metadata,Title,Publication title,Estatística do Movimento Fisiológico da Popula...,None,1925,PDF page 1 — Cover
1,Publication metadata,Reference year,Statistical reference year,1925,year,1925,PDF page 1 — Cover
2,Publication metadata,Publication year,Printed publication year,1929,year,1929,PDF page 1 — Imprint
3,Publication metadata,Publisher,Publication printer,Imprensa Nacional,None,1929,PDF page 1 — Imprint
4,Publication metadata,Institution,Responsible institution,Direcção Geral de Saúde — Portugal; Inspecção ...,None,1925,PDF page 1 — Cover
5,Index entry,Tabela I,Área e População recenseada (I-XII-1911 e I-XI...,None,None,1911 and 1920,PDF page 3 — Índice; printed page 1
6,Index entry,Tabela II,População recenseada (I-XII-1911 e I-XII-1920)...,None,None,1911 and 1920,PDF page 3 — Índice; printed pages 2 a 5
7,Index entry,Tabela III,"Casamentos, divórcios, nascimentos e óbitos, p...",None,None,None,PDF page 3 — Índice; printed pages 6 a 12
8,Index entry,Tabela XIV,"Óbitos, por idades e sexos, nos distritos",None,None,None,PDF page 3 — Índice; printed pages 70 e 71
9,Index entry,Tabela LVIII,"Taxas do movimento fisiológico, referidas à po...",None,None,None,PDF page 5 — Índice; printed pages 240 e 241


In [10]:
# ============================================================
# 9. Validate reference schema, counts and field types
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == EXPECTED_FIELDS
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

observed_category_counts = dict(
    Counter(
        reference_values_df["Category"]
    )
)

category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


type_issue_rows = []
missing_mandatory_rows = []

for row_index, row in reference_values_df.iterrows():

    for field in MANDATORY_STRING_FIELDS:
        value = row[field]

        if value is None or value == "":
            missing_mandatory_rows.append({
                "Record Index": int(row_index),
                "Field": field
            })

        elif not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": int(row_index),
                "Field": field,
                "Observed Type": type(value).__name__
            })

    for field in NULLABLE_STRING_FIELDS:
        value = row[field]

        if (
            value is not None
            and not isinstance(value, str)
        ):
            type_issue_rows.append({
                "Record Index": int(row_index),
                "Field": field,
                "Observed Type": type(value).__name__
            })

    value = row["Value"]

    if (
        isinstance(value, bool)
        or not isinstance(
            value,
            (str, int, float, type(None))
        )
    ):
        type_issue_rows.append({
            "Record Index": int(row_index),
            "Field": "Value",
            "Observed Type": type(value).__name__
        })


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_df = pd.DataFrame(missing_mandatory_rows)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_df.empty

print("Reference schema valid:", reference_schema_valid)
print("Record count valid:", record_count_valid)
print("Category counts valid:", category_counts_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
)

if not reference_schema_valid:
    raise AssertionError(
        "D9 reference schema is invalid."
    )

if not record_count_valid:
    raise AssertionError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} records, "
        f"found {len(reference_values_df)}."
    )

if not category_counts_valid:
    raise AssertionError(
        "D9 reference category counts are invalid."
    )

if not field_types_valid:
    display(type_issues_df)

    raise AssertionError(
        "D9 reference field types are invalid."
    )

if not mandatory_fields_complete:
    display(missing_mandatory_df)

    raise AssertionError(
        "D9 mandatory reference fields are incomplete."
    )


Reference schema valid: True
Record count valid: True
Category counts valid: True
Field types valid: True
Mandatory fields complete: True
{
  "Publication metadata": 5,
  "Index entry": 6,
  "Statistical value": 5,
  "Document structure": 3
}


In [11]:
# ============================================================
# 10. Duplicate and source-location integrity checks
# ============================================================

duplicate_mask = reference_values_df.duplicated(
    subset=EXPECTED_FIELDS,
    keep=False
)

duplicate_record_count = int(
    duplicate_mask.sum()
)

source_location_pattern_valid = bool(
    reference_values_df[
        "Source Location"
    ].map(
        lambda value: bool(
            re.match(
                r"^PDF page(?:s)? [1-8](?:[–-][1-8])?\b",
                value
            )
        )
    ).all()
)

source_values_expected = {
    "Portugal area": 91948.07,
    "Portugal population 1911": 5960056,
    "Portugal population 1920": 6032991,
    "Portugal density 1920": 65.6,
    "Portugal average annual population growth": 1.36
}

statistical_value_status = {}

for topic, expected_value in source_values_expected.items():
    matched_rows = reference_values_df[
        reference_values_df["Topic"] == topic
    ]

    statistical_value_status[topic] = (
        len(matched_rows) == 1
        and math.isclose(
            float(matched_rows.iloc[0]["Value"]),
            float(expected_value),
            rel_tol=1e-12,
            abs_tol=1e-12
        )
    )

statistical_values_valid = all(
    statistical_value_status.values()
)

period_distinction_valid = all([
    (
        reference_values_df.loc[
            reference_values_df["Topic"] == "Reference year",
            "Value"
        ].iloc[0] == 1925
    ),
    (
        reference_values_df.loc[
            reference_values_df["Topic"] == "Publication year",
            "Value"
        ].iloc[0] == 1929
    )
])

# ------------------------------------------------------------
# D9 field-semantic integrity checks
# ------------------------------------------------------------

index_value_null_valid = bool(
    reference_values_df.loc[
        reference_values_df["Category"] == "Index entry",
        "Value"
    ].isna().all()
)

document_structure_value_null_valid = bool(
    reference_values_df.loc[
        reference_values_df["Category"] == "Document structure",
        "Value"
    ].isna().all()
)

document_structure_period_null_valid = bool(
    reference_values_df.loc[
        reference_values_df["Category"] == "Document structure",
        "Reporting Period"
    ].isna().all()
)

index_period_rules_valid = all([
    (
        reference_values_df.loc[
            reference_values_df["Topic"] == "Tabela I",
            "Reporting Period"
        ].iloc[0] == "1911 and 1920"
    ),
    (
        reference_values_df.loc[
            reference_values_df["Topic"] == "Tabela II",
            "Reporting Period"
        ].iloc[0] == "1911 and 1920"
    ),
    reference_values_df.loc[
        reference_values_df["Topic"].isin([
            "Tabela III",
            "Tabela XIV",
            "Tabela LVIII",
            "Tabela LIX"
        ]),
        "Reporting Period"
    ].isna().all()
])

portugal_area_period_null_valid = bool(
    pd.isna(
        reference_values_df.loc[
            reference_values_df["Topic"] == "Portugal area",
            "Reporting Period"
        ].iloc[0]
    )
)

field_semantics_valid = all([
    index_value_null_valid,
    document_structure_value_null_valid,
    document_structure_period_null_valid,
    index_period_rules_valid,
    portugal_area_period_null_valid
])

print("Index-entry Values null:", index_value_null_valid)
print(
    "Document-structure Values null:",
    document_structure_value_null_valid
)
print(
    "Document-structure periods null:",
    document_structure_period_null_valid
)
print("Index period rules valid:", index_period_rules_valid)
print(
    "Portugal-area period null:",
    portugal_area_period_null_valid
)
print("Field semantics valid:", field_semantics_valid)

print("Duplicate records:", duplicate_record_count)
print("Source-location pattern valid:", source_location_pattern_valid)
print("Statistical values valid:", statistical_values_valid)
print("Reference/publication years distinct:", period_distinction_valid)


print(
    json.dumps(
        statistical_value_status,
        ensure_ascii=False,
        indent=2
    )
)

if duplicate_record_count != 0:
    display(
        reference_values_df.loc[
            duplicate_mask
        ]
    )

    raise AssertionError(
        "Unexpected duplicate D9 reference records detected."
    )

if not source_location_pattern_valid:
    raise AssertionError(
        "One or more D9 source locations have an invalid format."
    )

if not statistical_values_valid:
    raise AssertionError(
        "One or more fixed D9 statistical values are invalid."
    )

if not period_distinction_valid:
    raise AssertionError(
        "D9 reference year and publication year are not preserved distinctly."
    )

if not field_semantics_valid:
    raise AssertionError(
        "D9 reference field-semantic integrity checks failed."
    )

Index-entry Values null: True
Document-structure Values null: True
Document-structure periods null: True
Index period rules valid: True
Portugal-area period null: True
Field semantics valid: True
Duplicate records: 0
Source-location pattern valid: True
Statistical values valid: True
Reference/publication years distinct: True
{
  "Portugal area": true,
  "Portugal population 1911": true,
  "Portugal population 1920": true,
  "Portugal density 1920": true,
  "Portugal average annual population growth": true
}


In [14]:
# ============================================================
# 11. Document characterisation
# ============================================================

DOCUMENT_CHARACTERISATION = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "physical_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "native_text_characters": TOTAL_NATIVE_CHARACTERS,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "ocr_languages": ["Portuguese", "French"],
    "ocr_used_for_reference_values": False,
    "render_dpi": RENDER_DPI,
    "contains_cover_page": True,
    "contains_title_page": True,
    "contains_index_pages": True,
    "contains_historical_tables": True,
    "contains_rotated_tables": True,
    "rotated_pdf_pages": [7, 8],
    "contains_multilingual_headers": True,
    "represented_languages": [
        "Portuguese",
        "French"
    ],
    "contains_historical_typography": True,
    "contains_dense_numeric_tables": True,
    "represented_sections": [
        "Cover",
        "Title page",
        "Index",
        "Tabela I",
        "Tabela II"
    ],
    "fixed_extraction_scope": (
        "Five publication metadata records, six selected index entries, "
        "five selected Portugal-row values from Tabela I and three "
        "source-grounded structural records."
    ),
    "excluded_from_reference_scope": [
        "Index entries not selected in the fixed extraction task",
        "District-level and sex-specific values from Tabela I",
        "Age-by-sex values from Tabela II",
        "Values requiring OCR interpretation without manual confirmation",
        "Derived totals or rates",
        "French translations as separate duplicate records"
    ],
    "reference_schema_version": "v2",
    "reference_task_version": "v2",

}
print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "source_file": "D9 - EMovimentoFisiológico1925.pdf",
  "source_file_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "source_format": "PDF",
  "physical_page_count": 8,
  "page_count_valid": true,
  "native_text_characters": 0,
  "text_extractable": false,
  "ocr_required": true,
  "ocr_languages": [
    "Portuguese",
    "French"
  ],
  "ocr_used_for_reference_values": false,
  "render_dpi": 300,
  "contains_cover_page": true,
  "contains_title_page": true,
  "contains_index_pages": true,
  "contains_historical_tables": true,
  "contains_rotated_tables": true,
  "rotated_pdf_pages": [
    7,
    8
  ],
  "contains_multilingual_headers": true,
  "represented_languages": [
    "Portuguese",
    "French"
  ],
  "contains_historical_typography": true,
  "contains_dense_numeric_tables": true,
  "represented_sections": [
    "Cover",
    

In [15]:
# ============================================================
# 12. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "High",
        "Evidence Source":
            "Rendered-page inspection + OCR diagnostics",
        "Justification":
            "The publication combines cover material, index pages, "
            "dense historical tables and rotated table pages. "
            "Meaningful interpretation depends strongly on visual "
            "position and page orientation rather than on a simple "
            "linear reading sequence."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "High",
        "Evidence Source":
            "Manual table inspection + OCR diagnostics",
        "Justification":
            "Tabela I contains nested headings, district rows, "
            "sex-specific columns, totals, density measures and "
            "growth indicators, while Tabela II spans rotated pages. "
            "These relationships are difficult to preserve through "
            "plain text extraction."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Medium",
        "Evidence Source":
            "Manual source inspection",
        "Justification":
            "Major sections and table identifiers are visibly "
            "recognisable, but hierarchy depends partly on historical "
            "typography, bilingual headings and spatial organisation."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Medium",
        "Evidence Source":
            "Rendered-page inspection",
        "Justification":
            "The scanned pages remain visually interpretable, but "
            "fine historical typography and dense small numerical "
            "content reduce character-level clarity compared with "
            "born-digital documents."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Medium",
        "Evidence Source":
            "Rendered-page inspection + OCR results",
        "Justification":
            "Historical scan characteristics and typographic artefacts "
            "introduce OCR errors, although the document remains "
            "visually recoverable."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "High",
        "Evidence Source":
            "Automated PDF inspection",
        "Justification":
            "The PDF contains no meaningful native text layer, so "
            "machine-readable conversion depends on OCR."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual source inspection",
        "Justification":
            "Statistical terminology is internally coherent, but "
            "Portuguese labels appear alongside French translations "
            "and historical terminology and spelling conventions."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The fixed schema represents publication metadata, index "
            "entries, statistical values and structural observations, "
            "but these record types originate from substantially "
            "different source structures."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "High",
        "Evidence Source":
            "OCR diagnostics + manual table inspection",
        "Justification":
            "The statistical-table pages contain a high concentration "
            "of census counts, area values, density values and "
            "population statistics whose interpretation depends on "
            "correct row-column association."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Stage 1 verification",
        "Justification":
            "All information required by the predefined 19-record "
            "extraction task is visibly represented in the supplied "
            "eight-page source."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual source and reference inspection",
        "Justification":
            "The represented information is internally coherent, "
            "but several year concepts coexist, including statistical "
            "year 1925, publication year 1929 and census periods "
            "1911 and 1920, requiring careful contextual distinction."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "High",
        "Evidence Source":
            "Rendered-page profiling + manual inspection",
        "Justification":
            "The eight-page extract combines cover pages, bilingual "
            "index content, dense statistical tables, rotated pages "
            "and historical scanned typography."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "High",
        "Evidence Source":
            "Reference and source inspection",
        "Justification":
            "The document combines population counts, square "
            "kilometres, population-density measures, per-thousand "
            "growth rates, census dates, table identifiers, printed "
            "page references and bilingual labels."
    }
]

indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,High,Rendered-page inspection + OCR diagnostics,"The publication combines cover material, index..."
1,Structural Readiness,Table Structure Integrity,High,Manual table inspection + OCR diagnostics,"Tabela I contains nested headings, district ro..."
2,Structural Readiness,Section/Header Hierarchy,Medium,Manual source inspection,Major sections and table identifiers are visib...
3,Visual/OCR Readiness,Sharpness,Medium,Rendered-page inspection,The scanned pages remain visually interpretabl...
4,Visual/OCR Readiness,Noise / Degradation,Medium,Rendered-page inspection + OCR results,Historical scan characteristics and typographi...
5,Visual/OCR Readiness,OCR Dependency,High,Automated PDF inspection,The PDF contains no meaningful native text lay...
6,Semantic Quality,Terminology Consistency,Medium,Manual source inspection,Statistical terminology is internally coherent...
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,The fixed schema represents publication metada...
8,Semantic Quality,Numerical Density,High,OCR diagnostics + manual table inspection,The statistical-table pages contain a high con...
9,Completeness and Consistency,Required Field Presence,Low,Stage 1 verification,All information required by the predefined 19-...


In [16]:
# ============================================================
# 13. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )

if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [17]:
# ============================================================
# 14. Derive dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.50,2,Medium
1,Representation and Normalisation Complexity,3.00,2,High
2,Semantic Quality,2.33,3,Medium
3,Structural Readiness,2.67,3,High
4,Visual/OCR Readiness,2.33,3,Medium


In [18]:
# ============================================================
# 15. Build structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [19]:
# ============================================================
# 16. Reference summary and integrity report
# ============================================================

numeric_value_record_count = int(
    reference_values_df["Value"].map(
        lambda value: (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
        )
    ).sum()
)

text_value_record_count = int(
    reference_values_df["Value"].map(
        lambda value: isinstance(value, str)
    ).sum()
)

null_value_record_count = int(
    reference_values_df["Value"].isna().sum()
)

source_pages_represented = sorted(
    set(
        int(page)
        for location in reference_values_df[
            "Source Location"
        ]
        for page in re.findall(
            r"\b[1-8]\b",
            location.split("—")[0]
        )
    )
)

REFERENCE_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "reference_record_count": int(
        len(reference_values_df)
    ),
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "record_count_valid": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_valid": category_counts_valid,
    "numeric_value_record_count": numeric_value_record_count,
    "text_value_record_count": text_value_record_count,
    "null_value_record_count": null_value_record_count,
    "source_pages_represented": source_pages_represented,
    "fields": EXPECTED_FIELDS
}


REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "physical_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "ocr_dependency_confirmed": ocr_dependency_confirmed,
    "reference_schema_valid": reference_schema_valid,
    "record_count_valid": record_count_valid,
    "category_counts_valid": category_counts_valid,
    "field_types_valid": field_types_valid,
    "mandatory_fields_complete": mandatory_fields_complete,
    "duplicate_record_count": duplicate_record_count,
    "source_location_pattern_valid": source_location_pattern_valid,
    "statistical_values_valid": statistical_values_valid,
    "period_distinction_valid": period_distinction_valid,
    "manual_reference_construction": True,
    "ocr_used_for_reference_values": False,
    "calculation_applied": False,
    "inference_applied": False,
    "unit_conversion_applied": False,
    "source_value_repair_applied": False,
    "index_entry_values_null":
        index_value_null_valid,

    "document_structure_values_null":
        document_structure_value_null_valid,

    "document_structure_periods_null":
        document_structure_period_null_valid,

    "index_period_rules_valid":
        index_period_rules_valid,

    "portugal_area_period_null":
        portugal_area_period_null_valid,

    "field_semantics_valid":
        field_semantics_valid,
    "reference_integrity_passed": all([
        field_semantics_valid,
        PAGE_COUNT_VALID,
        OCR_REQUIRED,
        ocr_dependency_confirmed,
        reference_schema_valid,
        record_count_valid,
        category_counts_valid,
        field_types_valid,
        mandatory_fields_complete,
        duplicate_record_count == 0,
        source_location_pattern_valid,
        statistical_values_valid,
        period_distinction_valid
    ])
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "reference_record_count": 19,
  "expected_record_count": 19,
  "record_count_valid": true,
  "expected_category_counts": {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
  },
  "observed_category_counts": {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
  },
  "category_counts_valid": true,
  "numeric_value_record_count": 7,
  "text_value_record_count": 3,
  "null_value_record_count": 9,
  "source_pages_represented": [
    1,
    3,
    5,
    6,
    7,
    8
  ],
  "fields": [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
  ]
}
{
  "document_id": "D9",
  "source_file": "D9 - EMovimentoFisiológico1925.pdf",
  "source_file_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9

In [20]:
# ============================================================
# 17. Reference metadata
# ============================================================

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        "D9_reference_values.csv",

    "reference_construction_method":
        (
            "Manual document-grounded construction "
            "from rendered source pages"
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        EXPECTED_FIELDS,

    "ocr_used_for_diagnostics":
        True,

    "ocr_used_as_reference_source":
        False,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_value_repair_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],


    "notes":
    (
        "D9 is an image-based historical PDF. OCR is used only "
        "for diagnostic characterisation. Reference records were "
        "manually verified against the rendered source pages. "
        "A field-semantic integrity audit corrected cases where "
        "descriptive or document-level contextual information had "
        "been assigned to Value or Reporting Period despite no "
        "distinct record-level value or period being explicitly "
        "represented. The corrected reference dataset is frozen "
        "and reused unchanged across Branches A, B and C."
    )
}

In [21]:
# ============================================================
# 18. Export Stage 1 outputs
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_records,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SCHEMA_PATH.write_text(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SUMMARY_PATH.write_text(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_INTEGRITY_PATH.write_text(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

EXTRACTION_SCHEMA_PATH.write_text(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_METADATA_PATH.write_text(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

DOCUMENT_CHARACTERISATION_PATH.write_text(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

QUALITY_EVIDENCE_PATH.write_text(
    json.dumps(
        QUALITY_EVIDENCE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

OCR_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        OCR_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

ocr_results_df.to_csv(
    OCR_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

native_page_df.to_csv(
    NATIVE_PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

image_metadata_df.to_csv(
    IMAGE_METADATA_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("D9 Stage 1 outputs exported.")


D9 Stage 1 outputs exported.


In [22]:
# ============================================================
# 19. Final checks and output listing
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    REFERENCE_SCHEMA_PATH,
    EXTRACTION_SCHEMA_PATH,
    EXTRACTION_TASK_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    NATIVE_PAGE_CHARACTERISATION_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,
    OCR_RESULTS_PATH,
    IMAGE_METADATA_PATH,
    OCR_DIAGNOSTICS_PATH
]

missing_outputs = [
    path.name
    for path in GENERATED_OUTPUTS
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing D9 Stage 1 outputs: {missing_outputs}"
    )

if not REFERENCE_INTEGRITY[
    "reference_integrity_passed"
]:
    raise AssertionError(
        "D9 reference-integrity checks failed."
    )


print(
    "D9 Stage 1 completed successfully."
)

print()
print("Generated files:")

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


D9 Stage 1 completed successfully.

Generated files:
- D9_reference_values.csv | exists: True
- D9_reference_values.json | exists: True
- D9_reference_schema.json | exists: True
- D9_extraction_schema.json | exists: True
- D9_extraction_task.txt | exists: True
- D9_reference_summary.json | exists: True
- D9_reference_metadata.json | exists: True
- D9_reference_integrity.json | exists: True
- D9_document_characterisation.json | exists: True
- D9_native_page_characterisation.csv | exists: True
- D9_indicator_assessment.csv | exists: True
- D9_dimension_assessment.csv | exists: True
- D9_quality_evidence.json | exists: True
- D9_ocr_results.csv | exists: True
- D9_image_metadata.csv | exists: True
- D9_ocr_diagnostics.json | exists: True
